# **Start Section:**


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!pip install git+https://github.com/DaneshSelwal/treeffuser.git


In [ ]:
!pip install --quiet numpy==1.26.4 pandas scipy scikit-learn matplotlib seaborn xlsxwriter openpyxl torch properscoring


In [ ]:
import os
os.environ["PIP_CONSTRAINT"] = "/tmp/numpy_constraint.txt"
!echo "numpy==1.26.4" > /tmp/numpy_constraint.txt


# **Imports**


In [ ]:
import io
import os
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import properscoring as ps
import torch
import torch.nn as nn
import torch.optim as optim
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader


In [ ]:
# Go to find & replace button and replace (Data_folder) with your folder name. Rename your train and test dataset as train.csv and test.csv.
# Modify the names of the feature in the next cell if your input features are different.
target_column = None  # Keep None to automatically use the last column as the target.
random_seed = 42


In [ ]:
feature_names = ['Qt', 'Qt-1', 'St-1']


In [ ]:
train_data_path = "./drive/MyDrive/Data_folder/Data/train.csv"
test_data_path = "./drive/MyDrive/Data_folder/Data/test.csv"
output_folder = "./drive/MyDrive/Data_folder/Hyperspherical_Confidence_Mapping(HCM)"


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def build_demo_dataframe(n_rows, feature_names, seed=42, is_train=True):
    rng = np.random.default_rng(seed + (0 if is_train else 99))
    data = {name: rng.normal(loc=0.0, scale=1.0, size=n_rows) for name in feature_names}
    frame = pd.DataFrame(data)
    nonlinear_term = 0.8 * np.sin(frame[feature_names[0]].values)
    interaction_term = 0.5 * frame[feature_names[1]].values * frame[feature_names[2]].values
    trend_term = 0.3 * frame[feature_names[0]].values ** 2
    noise = rng.normal(loc=0.0, scale=0.35 if is_train else 0.4, size=n_rows)
    frame['Target'] = 12.0 + 3.2 * frame[feature_names[0]].values - 1.7 * frame[feature_names[1]].values + 2.1 * frame[feature_names[2]].values + nonlinear_term + interaction_term - trend_term + noise
    return frame


def read_csv_from_candidates(path_candidates):
    for path in path_candidates:
        path = Path(path)
        if not path.exists():
            continue
        try:
            df = pd.read_csv(path)
            if not df.empty:
                print(f"Loaded data from: {path}")
                return df
        except Exception:
            continue
    return pd.DataFrame()


set_seed(random_seed)
train_demo = build_demo_dataframe(240, feature_names, seed=random_seed, is_train=True)
test_demo = build_demo_dataframe(80, feature_names, seed=random_seed, is_train=False)

train_candidates = [
    train_data_path,
    "/content/drive/MyDrive/Data_folder/Data/train.csv",
    "Data_folder/Data/train.csv"
]
test_candidates = [
    test_data_path,
    "/content/drive/MyDrive/Data_folder/Data/test.csv",
    "Data_folder/Data/test.csv"
]

train_data = read_csv_from_candidates(train_candidates)
test_data = read_csv_from_candidates(test_candidates)

if train_data.empty:
    print("Warning: training CSV is missing or empty. A generated demo dataset will be used so the notebook can still run in Google Colab.")
    train_data = train_demo.copy()
if test_data.empty:
    print("Warning: testing CSV is missing or empty. A generated demo dataset will be used so the notebook can still run in Google Colab.")
    test_data = test_demo.copy()


In [ ]:
print("\nShape of training data:", train_data.shape)
print("First 5 rows of training data:\n", train_data.head(5))
print("\nShape of test data:", test_data.shape)
print("First 5 rows of test data:\n", test_data.head(5))


In [ ]:
if target_column is None:
    target_column = train_data.columns[-1]

available_feature_names = [name for name in feature_names if name in train_data.columns]
if len(available_feature_names) == len(feature_names):
    selected_feature_names = feature_names
else:
    selected_feature_names = train_data.columns[:-1].tolist()
    print(f"Feature names were adjusted automatically to match the dataset columns: {selected_feature_names}")

X_train_full = train_data[selected_feature_names].copy()
y_train_full = train_data[target_column].copy()
X_test = test_data[selected_feature_names].copy()
y_test = test_data[target_column].copy()


In [ ]:
# Apply z-score normalization
X_train_raw, X_val_raw, y_train_raw, y_val_raw = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.20,
    random_state=random_seed
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_val = scaler.transform(X_val_raw)
X_test_scaled = scaler.transform(X_test)

y_train = y_train_raw.to_numpy(dtype=np.float32).reshape(-1, 1)
y_val = y_val_raw.to_numpy(dtype=np.float32).reshape(-1, 1)
y_test_array = y_test.to_numpy(dtype=np.float32).reshape(-1, 1)

print("\nSelected feature names:", selected_feature_names)
print("Target column:", target_column)
print("Training split:", X_train.shape, y_train.shape)
print("Validation split:", X_val.shape, y_val.shape)
print("Test split:", X_test_scaled.shape, y_test_array.shape)


# **Functions:**


In [ ]:
def expand_scalar_targets(y_array):
    y_array = np.asarray(y_array, dtype=np.float32).reshape(-1, 1)
    return np.concatenate([y_array, y_array], axis=1)


def create_dataloader(X_array, y_array, batch_size=64, shuffle=True):
    features = torch.tensor(X_array, dtype=torch.float32)
    expanded_targets = torch.tensor(expand_scalar_targets(y_array), dtype=torch.float32)
    dataset = TensorDataset(features, expanded_targets)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


class HCMUCIRegressor(nn.Module):
    """Paper-style tabular regression backbone: 3 hidden layers of width 20 with LeakyReLU."""

    def __init__(self, input_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 20)
        self.fc2 = nn.Linear(20, 20)
        self.fc3 = nn.Linear(20, 20)
        self.fc4 = nn.Linear(20, 2)   # direction d
        self.fc5 = nn.Linear(20, 1)   # magnitude R
        self.act = nn.LeakyReLU(0.01)

    def forward(self, x):
        x = self.act(self.fc1(x))
        x = self.act(self.fc2(x))
        x = self.act(self.fc3(x))
        d = self.fc4(x)
        R = self.fc5(x)
        return R, d


def train_hcm_model(X_train, y_train, X_val, y_val, training_config):
    set_seed(training_config['seed'])
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = HCMUCIRegressor(X_train.shape[1]).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(
        model.parameters(),
        lr=training_config['learning_rate'],
        weight_decay=training_config['weight_decay']
    )

    train_loader = create_dataloader(X_train, y_train, batch_size=training_config['batch_size'], shuffle=True)
    history = []
    best_state = None
    best_val_loss = np.inf
    patience_counter = 0

    y_val_expanded = expand_scalar_targets(y_val)

    for epoch in range(training_config['epochs']):
        model.train()
        epoch_loss = []
        epoch_r_loss = []
        epoch_d_loss = []

        for x_batch, y_expanded in train_loader:
            x_batch = x_batch.to(device)
            y_expanded = y_expanded.to(device)

            optimizer.zero_grad()
            pred_R, pred_d = model(x_batch)

            R_target = torch.sqrt(torch.sum(y_expanded ** 2, dim=1, keepdim=True))
            d_target = y_expanded / (R_target + 1e-8)

            d_loss = criterion(pred_R * d_target, y_expanded)
            R_loss = criterion(R_target * pred_d, y_expanded)
            loss = d_loss + R_loss

            loss.backward()
            optimizer.step()

            epoch_loss.append(loss.item())
            epoch_r_loss.append(R_loss.item())
            epoch_d_loss.append(d_loss.item())

        val_outputs = predict_hcm_outputs(model, X_val, device)
        val_loss = mean_squared_error(y_val.reshape(-1), val_outputs['mean_prediction'])
        history.append({
            'epoch': epoch + 1,
            'train_loss': float(np.mean(epoch_loss)),
            'R_loss': float(np.mean(epoch_r_loss)),
            'd_loss': float(np.mean(epoch_d_loss)),
            'val_mse': float(val_loss)
        })

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if (epoch + 1) % training_config['print_every'] == 0 or epoch == 0:
            print(f"Epoch {epoch + 1}/{training_config['epochs']} | Loss: {np.mean(epoch_loss):.5f} | R Loss: {np.mean(epoch_r_loss):.5f} | d Loss: {np.mean(epoch_d_loss):.5f} | Val MSE: {val_loss:.5f}")

        if patience_counter >= training_config['patience']:
            print(f"Early stopping triggered at epoch {epoch + 1}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, pd.DataFrame(history), device


@torch.no_grad()
def predict_hcm_outputs(model, X_array, device):
    model.eval()
    model.to(device)

    x_tensor = torch.tensor(X_array, dtype=torch.float32, device=device)
    pred_R, pred_d = model(x_tensor)

    d_norm_sq = torch.sum(pred_d ** 2, dim=1)
    d_norm = torch.sqrt(torch.clamp(d_norm_sq, min=1e-12))

    # Public GitHub regression example uses this as the uncertainty band width.
    sigma_hat = torch.sqrt(torch.abs(d_norm_sq - 1.0)) * torch.abs(pred_R.squeeze(-1))
    raw_u = torch.abs(pred_R.squeeze(-1)) * torch.abs(d_norm - 1.0)
    pred_y = (pred_R * pred_d)[:, 0]

    return {
        'mean_prediction': pred_y.cpu().numpy(),
        'pred_R': pred_R.squeeze(-1).cpu().numpy(),
        'pred_d': pred_d.cpu().numpy(),
        'd_norm_sq': d_norm_sq.cpu().numpy(),
        'd_norm': d_norm.cpu().numpy(),
        'sigma_hat_raw': sigma_hat.cpu().numpy(),
        'raw_uncertainty_score': raw_u.cpu().numpy(),
    }


In [ ]:
def search_temperature_scale(raw_sigma, absolute_errors, grid=None):
    raw_sigma = np.clip(np.asarray(raw_sigma, dtype=float), 1e-8, None)
    absolute_errors = np.asarray(absolute_errors, dtype=float)

    if grid is None:
        grid = np.logspace(-2, 2, 400)

    target_coverage = np.array([0.68, 0.95, 0.997])
    best_scale = 1.0
    best_loss = np.inf

    for scale in grid:
        sigma_scaled = raw_sigma * scale
        coverage = np.array([
            np.mean(absolute_errors <= sigma_scaled),
            np.mean(absolute_errors <= 2.0 * sigma_scaled),
            np.mean(absolute_errors <= 3.0 * sigma_scaled)
        ])
        loss = np.sum((coverage - target_coverage) ** 2)
        if loss < best_loss:
            best_loss = loss
            best_scale = scale

    return float(best_scale)


def normalize_confidence(confidence_values):
    confidence_values = np.asarray(confidence_values, dtype=float)
    cmin, cmax = confidence_values.min(), confidence_values.max()
    if np.isclose(cmin, cmax):
        return np.ones_like(confidence_values)
    return (confidence_values - cmin) / (cmax - cmin)


def create_predictions_dataframe(output_dict, y_true, sigma_scale):
    y_true = np.asarray(y_true).reshape(-1)
    mean_prediction = output_dict['mean_prediction']
    sigma_raw = np.clip(output_dict['sigma_hat_raw'], 1e-8, None)
    sigma_calibrated = sigma_raw * sigma_scale
    absolute_error = np.abs(y_true - mean_prediction)

    confidence = np.exp(-sigma_calibrated)
    confidence_normalized = normalize_confidence(confidence)

    predictions_df = pd.DataFrame({
        'Mean': mean_prediction,
        'Sigma_Hat_Raw': sigma_raw,
        'Sigma_Hat': sigma_calibrated,
        'StdDev': sigma_calibrated,
        'Confidence': confidence,
        'Confidence_Normalized': confidence_normalized,
        'RawUncertainty': output_dict['raw_uncertainty_score'],
        'Magnitude_R': output_dict['pred_R'],
        'Direction_Norm': output_dict['d_norm'],
        'Direction_Norm_Squared': output_dict['d_norm_sq'],
        'Absolute_Error': absolute_error,
        'Actual': y_true
    })

    for k in [1, 2, 3]:
        predictions_df[f'Lower_{k}Sigma'] = predictions_df['Mean'] - k * predictions_df['Sigma_Hat']
        predictions_df[f'Upper_{k}Sigma'] = predictions_df['Mean'] + k * predictions_df['Sigma_Hat']
        predictions_df[f'Coverage_{k}Sigma'] = (
            (predictions_df['Actual'] >= predictions_df[f'Lower_{k}Sigma']) &
            (predictions_df['Actual'] <= predictions_df[f'Upper_{k}Sigma'])
        ).astype(int)

    return predictions_df


def compute_regression_ece(sigma_values, absolute_errors, n_bins=10):
    sigma_values = np.asarray(sigma_values, dtype=float)
    absolute_errors = np.asarray(absolute_errors, dtype=float)
    quantile_edges = np.quantile(sigma_values, np.linspace(0, 1, n_bins + 1))
    quantile_edges[0] -= 1e-8
    quantile_edges[-1] += 1e-8

    total = len(sigma_values)
    ece = 0.0
    rows = []
    for idx in range(n_bins):
        lower = quantile_edges[idx]
        upper = quantile_edges[idx + 1]
        mask = (sigma_values >= lower) & (sigma_values < upper)
        if not np.any(mask):
            continue
        mean_sigma = sigma_values[mask].mean()
        mean_error = absolute_errors[mask].mean()
        weight = mask.sum() / total
        ece += weight * abs(mean_sigma - mean_error)
        rows.append({
            'Bin': idx + 1,
            'Bin_Size': int(mask.sum()),
            'Mean_Sigma': mean_sigma,
            'Mean_Absolute_Error': mean_error,
            'Absolute_Gap': abs(mean_sigma - mean_error)
        })
    return float(ece), pd.DataFrame(rows)


def compute_paper_metrics(predictions_df, model_name='HCM'):
    absolute_error = predictions_df['Absolute_Error'].values
    sigma = np.clip(predictions_df['Sigma_Hat'].values, 1e-8, None)

    coverage_1 = np.mean(absolute_error <= sigma)
    coverage_2 = np.mean(absolute_error <= 2.0 * sigma)
    coverage_3 = np.mean(absolute_error <= 3.0 * sigma)
    ece_reg, ece_bins_df = compute_regression_ece(sigma, absolute_error, n_bins=10)

    pearson_value = pearsonr(sigma, absolute_error)[0]
    spearman_value = spearmanr(sigma, absolute_error)[0]
    rmse = np.sqrt(mean_squared_error(predictions_df['Actual'], predictions_df['Mean']))
    mae = mean_absolute_error(predictions_df['Actual'], predictions_df['Mean'])

    summary_df = pd.DataFrame([{
        'Model': model_name,
        'Cov@1Sigma': coverage_1,
        'Cov@2Sigma': coverage_2,
        'Cov@3Sigma': coverage_3,
        'Pearson': pearson_value,
        'Spearman': spearman_value,
        'ECE_reg': ece_reg,
        'RMSE': rmse,
        'MAE': mae
    }])
    return summary_df, ece_bins_df


In [ ]:
def insert_figure_into_worksheet(writer, sheet_name, figure, image_cell='H2'):
    image_buffer = io.BytesIO()
    figure.savefig(image_buffer, format='png', dpi=200, bbox_inches='tight')
    image_buffer.seek(0)
    worksheet = writer.sheets[sheet_name]
    worksheet.insert_image(image_cell, 'plot.png', {'image_data': image_buffer})
    plt.close(figure)


def generate_and_save_plots(predictions_df, excel_path):
    with pd.ExcelWriter(excel_path, engine='xlsxwriter') as writer:
        predictions_df.to_excel(writer, sheet_name='Predictions', index=False)

        # Paper-style ordered band plot with 1σ, 2σ, 3σ intervals.
        ordered_df = predictions_df.sort_values('Actual').reset_index(drop=True)
        fig1, ax1 = plt.subplots(figsize=(12, 8))
        ax1.plot(ordered_df.index, ordered_df['Mean'], color='black', label='Prediction', linewidth=1.2)
        ax1.scatter(ordered_df.index, ordered_df['Actual'], color='darkorange', s=18, alpha=0.8, label='Actual')
        for k, alpha in zip([3, 2, 1], [0.10, 0.16, 0.24]):
            ax1.fill_between(
                ordered_df.index,
                ordered_df[f'Lower_{k}Sigma'],
                ordered_df[f'Upper_{k}Sigma'],
                color='skyblue',
                alpha=alpha,
                label=f'{k}σ interval'
            )
        ax1.set_title('HCM Prediction with 1σ, 2σ, 3σ Bands')
        ax1.set_xlabel('Ordered Test Sample')
        ax1.set_ylabel('Target Value')
        ax1.legend(loc='best')
        ordered_df.to_excel(writer, sheet_name='Bands_1_2_3Sigma', index=False)
        insert_figure_into_worksheet(writer, 'Bands_1_2_3Sigma', fig1)

        fig2, ax2 = plt.subplots(figsize=(10, 7))
        sns.scatterplot(x='Sigma_Hat', y='Absolute_Error', data=predictions_df, ax=ax2)
        ax2.set_title('HCM Uncertainty vs Absolute Error')
        ax2.set_xlabel('Calibrated Sigma_Hat')
        ax2.set_ylabel('Absolute Error')
        pd.DataFrame({'Sigma_Hat': predictions_df['Sigma_Hat'], 'Absolute_Error': predictions_df['Absolute_Error']}).to_excel(writer, sheet_name='Uncertainty_vs_Error', index=False)
        insert_figure_into_worksheet(writer, 'Uncertainty_vs_Error', fig2)

        fig3, ax3 = plt.subplots(figsize=(10, 7))
        sns.histplot(predictions_df['Confidence_Normalized'], bins=20, kde=True, ax=ax3)
        ax3.set_title('Distribution of Normalized Confidence')
        ax3.set_xlabel('Normalized Confidence')
        ax3.set_ylabel('Frequency')
        pd.DataFrame({'Confidence_Normalized': predictions_df['Confidence_Normalized']}).to_excel(writer, sheet_name='Confidence_Distribution', index=False)
        insert_figure_into_worksheet(writer, 'Confidence_Distribution', fig3)

        fig4, ax4 = plt.subplots(figsize=(10, 7))
        sns.scatterplot(x='Actual', y='Mean', data=predictions_df, ax=ax4)
        diagonal_min = min(predictions_df['Actual'].min(), predictions_df['Mean'].min())
        diagonal_max = max(predictions_df['Actual'].max(), predictions_df['Mean'].max())
        ax4.plot([diagonal_min, diagonal_max], [diagonal_min, diagonal_max], linestyle='--', color='red')
        ax4.set_title('Predicted Y_Label vs Actual Y_Label')
        ax4.set_xlabel('Actual Y_Label')
        ax4.set_ylabel('Predicted Mean Y_Label')
        pd.DataFrame({'Actual': predictions_df['Actual'], 'Mean': predictions_df['Mean']}).to_excel(writer, sheet_name='Predicted_vs_Actual', index=False)
        insert_figure_into_worksheet(writer, 'Predicted_vs_Actual', fig4)


In [ ]:
def generate_and_save_paper_metrics(predictions_df, excel_path, model_name='HCM'):
    paper_metrics_df, ece_bins_df = compute_paper_metrics(predictions_df, model_name=model_name)

    coverage_df = pd.DataFrame([
        {'Scale': '1Sigma', 'Coverage': predictions_df['Coverage_1Sigma'].mean(), 'Target': 0.68},
        {'Scale': '2Sigma', 'Coverage': predictions_df['Coverage_2Sigma'].mean(), 'Target': 0.95},
        {'Scale': '3Sigma', 'Coverage': predictions_df['Coverage_3Sigma'].mean(), 'Target': 0.997},
    ])

    confidence_bins = pd.qcut(predictions_df['Confidence_Normalized'], q=min(10, len(predictions_df)), duplicates='drop')
    confidence_bin_df = predictions_df.assign(Confidence_Bin=confidence_bins).groupby('Confidence_Bin', observed=False).agg(
        Mean_Confidence=('Confidence_Normalized', 'mean'),
        Mean_Absolute_Error=('Absolute_Error', 'mean'),
        Mean_Sigma=('Sigma_Hat', 'mean'),
        Count=('Confidence_Normalized', 'size')
    ).reset_index()

    with pd.ExcelWriter(excel_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
        paper_metrics_df.to_excel(writer, sheet_name='Paper_Metrics', index=False)
        coverage_df.to_excel(writer, sheet_name='Coverage_Details', index=False)
        ece_bins_df.to_excel(writer, sheet_name='ECE_Bins', index=False)
        confidence_bin_df.to_excel(writer, sheet_name='Confidence_Bins', index=False)


def generate_and_save_secondary_metrics(predictions_df, excel_path, model_name='HCM'):
    mu_pred = predictions_df['Mean'].values
    sigma_pred = np.clip(predictions_df['Sigma_Hat'].values, 1e-8, None)
    y_true = predictions_df['Actual'].values

    crps_values = ps.crps_gaussian(y_true, mu=mu_pred, sig=sigma_pred)
    nll_values = -0.5 * np.log(2 * np.pi * sigma_pred**2) - ((y_true - mu_pred) ** 2) / (2 * sigma_pred**2)
    secondary_df = pd.DataFrame({
        'Mean': mu_pred,
        'Sigma_Hat': sigma_pred,
        'CRPS': crps_values,
        'LogLikelihood': nll_values
    })

    summary_df = pd.DataFrame([{
        'Model': model_name,
        'Mean_CRPS': float(np.mean(crps_values)),
        'Mean_Negative_LogLikelihood': float(np.mean(-nll_values))
    }])

    with pd.ExcelWriter(excel_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
        secondary_df.to_excel(writer, sheet_name='Secondary_Compatibility', index=False)
        summary_df.to_excel(writer, sheet_name='Secondary_Summary', index=False)


In [ ]:
# Define the path to the folder
folder_path = "/content/drive/MyDrive/Data_folder"
if 'google.colab' in sys.modules:
    os.makedirs(folder_path, exist_ok=True)

if 'google.colab' in sys.modules:
    output_candidates = [
        Path(output_folder),
        Path('/content/drive/MyDrive/Data_folder/Hyperspherical_Confidence_Mapping(HCM)')
    ]
else:
    output_candidates = [
        Path('Data_folder/Hyperspherical_Confidence_Mapping(HCM)'),
        Path(output_folder)
    ]

for candidate in output_candidates:
    try:
        candidate.mkdir(parents=True, exist_ok=True)
        resolved_output_folder = candidate
        break
    except Exception:
        continue
else:
    raise RuntimeError('Could not create an output folder for HCM results.')


In [ ]:
def build_matrix_evaluation(predictions_df, model_name='HCM'):
    paper_metrics_df, _ = compute_paper_metrics(predictions_df, model_name=model_name)
    matrix_df = paper_metrics_df.copy()
    matrix_df['Mean_Sigma_Hat'] = predictions_df['Sigma_Hat'].mean()
    matrix_df['Mean_RawUncertainty'] = predictions_df['RawUncertainty'].mean()
    return matrix_df


# **Hyperspherical Confidence Mapping (HCM)**


In [ ]:
# Paper-style tabular regression setup from the HCM paper appendix.
training_config = {
    'seed': random_seed,
    'epochs': 200,
    'batch_size': 64,
    'learning_rate': 1e-4,
    'weight_decay': 1e-4,
    'patience': 40,
    'print_every': 20
}
training_config


In [ ]:
model, training_history_df, device = train_hcm_model(
    X_train,
    y_train,
    X_val,
    y_val,
    training_config
)

validation_outputs = predict_hcm_outputs(model, X_val, device)
validation_scale = search_temperature_scale(
    validation_outputs['sigma_hat_raw'],
    np.abs(y_val.reshape(-1) - validation_outputs['mean_prediction'])
)
print(f"Validation temperature scale: {validation_scale:.6f}")

test_outputs = predict_hcm_outputs(model, X_test_scaled, device)
predictions_HCM_df = create_predictions_dataframe(test_outputs, y_test_array, validation_scale)

print(predictions_HCM_df.head())


In [ ]:
hcm_excel_path = resolved_output_folder / 'HCM.xlsx'
generate_and_save_plots(predictions_HCM_df, hcm_excel_path)
print(f"Saved detailed HCM outputs to: {hcm_excel_path}")


In [ ]:
generate_and_save_paper_metrics(predictions_HCM_df, hcm_excel_path, model_name='HCM')
with pd.ExcelWriter(hcm_excel_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    training_history_df.to_excel(writer, sheet_name='Training_History', index=False)


In [ ]:
generate_and_save_secondary_metrics(predictions_HCM_df, hcm_excel_path, model_name='HCM')


# **Matrix Evaulation**


In [ ]:
matrix_evaluation_df = build_matrix_evaluation(predictions_HCM_df, model_name='HCM')
matrix_excel_path = resolved_output_folder / 'Matrix Evaluation.xlsx'
with pd.ExcelWriter(matrix_excel_path, engine='xlsxwriter') as writer:
    matrix_evaluation_df.to_excel(writer, sheet_name='Matrix Evaluation', index=False)

print(matrix_evaluation_df)
print(f"Saved matrix evaluation to: {matrix_excel_path}")
